# Data preparation of generation data from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Hourly generation per technology and country for selected year (generation_"+year+"_hourly_entsoe.csv) 
- Weekly generation per technology and country for selected year (generation_"+year+"_weekly_entsoe.csv) 
- Monthly generation per technology and country for selected year (generation_"+year+"_monthly_entsoe.csv) 
- Yearly generation per technology and country for selected year (generation_"+year+"_annual_entsoe.csv) 

Settings in next window

In [74]:
#download files again (yes/no)?
download = "yes"

#set year for data creation
year = '2024'

In [75]:
import pysftp
import sys
import os
import pandas as pd
import datetime as dt
import plotly.express as px

In [76]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [77]:
#cnopts = pysftp.CnOpts()
#cnopts.hostkeys = None

In [78]:
dir_out = "../parsed_data/"

In [79]:
#technology definition
dict_agg_tech = {"Other": "Other", #####
                "Wind Offshore": "WindOffshore", ####
                "Fossil Brown coal/Lignite": "Lignite", ####
                "Nuclear": "Nuclear", ####
                "Fossil Hard coal": "HardCoal", ####
                "Geothermal": "Other", ####
                "Fossil Coal-derived gas": "Other", #####
                "Hydro Pumped Storage": "Pump", ####
                "Hydro Run-of-river and poundage": "RunOfRiver", ####
                "Biomass": "Biomass", #####
                "Fossil Peat": "Other", ####
                "Fossil Oil shale": "Oil", #####
                "Fossil Oil": "Oil", ####
                "Hydro Water Reservoir": "Reservoir", ####
                "Marine": "Other", ####
                "Wind Onshore": "WindOnshore", #####
                "Other renewable": "Other", ###
                "Solar": "Solar", ####
                "Waste": "Other", ####
                "Fossil Gas": "Gas", ####
                "rooftop_pv" : "Solar", ## Leon, könnte Zeile weglassen, kommt doch nicht vor 
                "onshore_wind" : "WindOnshore", ## Leon, könnte Zeile weglassen, kommt doch nicht vor 
                "offshore_wind" : "WindOffshore"} ## Leon, könnte Zeile weglassen, kommt doch nicht vor 

In [80]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [81]:
# show list of all available folders (uncomment last line if needed)
# relevant folder was renamed to AggregatedGenerationPerType_16.1.B_C
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    files = sftp.listdir('/TP_export/')   
    print(files)

Connection succesfully established.
['AcceptedAggregatedOffers_17.1.D', 'ActivatedBalancingEnergy_17.1.E', 'ActualCapacitiesAndOutlookOnFrequencyRestorationReserveAndReplacementReserve_SOGL_188.3_188.4_189.2_189.3_r3', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r2.1', 'ActualTotalLoad_6.1.A', 'AggregatedBalancingEnergyBids_12.3.E_r3', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D_r3', 'AggregatedGenerationPerType_16.1.B_C', 'AmountAndPricesPaidOfBalancingReservesUnderContract_17.1.B_C_r2', 'AmountOfBalancingReservesUnderContract_17.1.B', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r2', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r3', 'ChangesInActualAvailabilityOfConsumptionUnits_7.1.B', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructureReasons_10.1.C', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructure_10.1.C', 'ChangesToBidAvailability_IFs_mFRR9.9_aFRR9.6_9.8_r3', 'Com

In [82]:
#load file names from server
path_gen = path+'AggregatedGenerationPerType_16.1.B_C/'
path_gen_local = path_local+'generation/'

# Leon: (Ordner sicherstellen)
os.makedirs(path_gen_local, exist_ok=True)

with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_gen)
    #download files
    if year != "":
        files = [i for i in files if year in i]
print(files)

Connection succesfully established.
['2024_01_AggregatedGenerationPerType_16.1.B_C.csv', '2024_02_AggregatedGenerationPerType_16.1.B_C.csv', '2024_03_AggregatedGenerationPerType_16.1.B_C.csv', '2024_04_AggregatedGenerationPerType_16.1.B_C.csv', '2024_05_AggregatedGenerationPerType_16.1.B_C.csv', '2024_06_AggregatedGenerationPerType_16.1.B_C.csv', '2024_07_AggregatedGenerationPerType_16.1.B_C.csv', '2024_08_AggregatedGenerationPerType_16.1.B_C.csv', '2024_09_AggregatedGenerationPerType_16.1.B_C.csv', '2024_10_AggregatedGenerationPerType_16.1.B_C.csv', '2024_11_AggregatedGenerationPerType_16.1.B_C.csv', '2024_12_AggregatedGenerationPerType_16.1.B_C.csv']


In [83]:
#download aggregated generation data (AggregatedGenerationPerType)
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_gen+file,path_gen_local+file)
            print('Successfully downloaded file '+file)

Successfully downloaded file 2024_01_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_02_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_03_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_04_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_05_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_06_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_07_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_08_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_09_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_10_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_11_AggregatedGenerationPerType_16.1.B_C.csv
Successfully downloaded file 2024_12_AggregatedGenerationPerType_16.1.B_C.csv


In [84]:
#Leon
import chardet
# Eine der Datein testen
test_file = path_gen_local + files[0]
print("Prüfe Datei:", test_file)

# Encoding automatisch erkennen
with open(test_file, 'rb') as f:
    result = chardet.detect(f.read(50000)) # nur ersten Teil lesen
print("Erkanntes Encoding:", result)

# Datei mit erkanntem Encoding laden
df_test = pd.read_csv(test_file, sep="\t", encoding=result['encoding'], nrows=5)
print("Spalten:", df_test.columns.tolist())



Prüfe Datei: ..\source_data/ENTSOE/generation/2024_01_AggregatedGenerationPerType_16.1.B_C.csv
Erkanntes Encoding: {'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': ''}
Spalten: ['DateTime', 'ResolutionCode', 'AreaCode', 'AreaTypeCode', 'AreaName', 'MapCode', 'ProductionType', 'ActualGenerationOutput', 'ActualConsumption', 'UpdateTime']


In [85]:
#combine files to one data frame
df_gen_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_gen_local+file,
                          decimal=".",encoding="UTF-8-SIG",sep="\t",  # angepasstes encoding (Leon)
                          parse_dates=True, index_col="DateTime")
    #df_gen_in = df_gen_in.append(df_temp)
    df_gen_in = pd.concat([df_gen_in, df_temp], ignore_index=False) # Leon, in pandas 2.0 gibt es .append() nicht mehr als Methode
    
#country values for HR are missing so we change this to area type code
df_gen_in.loc[((df_gen_in.MapCode == 'HR') & (df_gen_in.AreaTypeCode == 'CTY')),'AreaTypeCode'] = 'not_it'
df_gen_in.loc[((df_gen_in.MapCode == 'HR') & (df_gen_in.AreaTypeCode == 'BZN')),'AreaTypeCode'] = 'CTY'
df_gen_in = df_gen_in[df_gen_in.AreaTypeCode == "CTY"].drop(["AreaCode","AreaTypeCode"], axis=1).reset_index()
df_gen_in = df_gen_in.sort_values(by=['DateTime'])
df_gen_in["technology"] = df_gen_in.ProductionType.map(dict_agg_tech) # TECH - MAPPING 
df_gen_in.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6394713 entries, 2418 to 6388814
Data columns (total 9 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   DateTime                datetime64[ns]
 1   ResolutionCode          object        
 2   AreaName                object        
 3   MapCode                 object        
 4   ProductionType          object        
 5   ActualGenerationOutput  float64       
 6   ActualConsumption       float64       
 7   UpdateTime              object        
 8   technology              object        
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 487.9+ MB


In [97]:
df_gen_in.tail(40) # Leon 

,DateTime,MapCode,ActualGenerationOutput,ActualConsumption,technology
6388796,2024-12-31 23:45:00,CZ,0.00,209.43,Pump
6388538,2024-12-31 23:45:00,FR,4407.10,NaN,RunOfRiver
6388534,2024-12-31 23:45:00,FR,1832.28,NaN,Gas
6388530,2024-12-31 23:45:00,FR,44708.22,NaN,Nuclear
6388810,2024-12-31 23:45:00,CZ,23.10,NaN,Oil
6388526,2024-12-31 23:45:00,FR,1370.14,NaN,WindOffshore
6388522,2024-12-31 23:45:00,PL,56.38,NaN,Other
6388518,2024-12-31 23:45:00,PL,4992.60,NaN,HardCoal
6389008,2024-12-31 23:45:00,DE,2714.25,NaN,HardCoal
6388514,2024-12-31 23:45:00,PL,31.50,NaN,Reservoir


In [87]:
df_temp = pd.read_csv(path_gen_local+files[10],
                      decimal=".",encoding="UTF-8-SIG",sep="\t",
                      parse_dates=True, index_col="DateTime")
df_temp.head()

,ResolutionCode,AreaCode,AreaTypeCode,AreaName,MapCode,ProductionType,ActualGenerationOutput,ActualConsumption,UpdateTime
DateTime,,,,,,,,,
2024-11-02 06:15:00,PT15M,10YES-REE------0,CTA,ES CTA,ES,Hydro Water Reservoir,2356.0,NaN,2024-11-02 07:35:24.024
2024-11-02 06:15:00,PT15M,10YES-REE------0,CTA,ES CTA,ES,Hydro Pumped Storage,860.0,720.0,2024-11-02 07:35:24.024
2024-11-02 06:15:00,PT15M,10YES-REE------0,CTA,ES CTA,ES,Marine,0.0,NaN,2024-11-02 07:35:24.024
2024-11-02 06:15:00,PT15M,10YES-REE------0,CTA,ES CTA,ES,Geothermal,0.0,NaN,2024-11-02 07:35:25.025
2024-11-02 06:15:00,PT15M,10YES-REE------0,CTA,ES CTA,ES,Fossil Oil,20.0,NaN,2024-11-02 07:35:25.025


In [88]:
df_temp['ProductionType'].unique()

array(['Hydro Water Reservoir', 'Hydro Pumped Storage', 'Marine',
       'Geothermal', 'Fossil Oil', 'Nuclear', 'Fossil Gas',
       'Fossil Coal-derived gas', 'Fossil Oil shale', 'Other', 'Waste',
       'Other renewable', 'Wind Offshore', 'Fossil Brown coal/Lignite',
       'Biomass', 'Fossil Hard coal', 'Hydro Run-of-river and poundage',
       'Fossil Peat', 'Wind Onshore', 'Solar'], dtype=object)

In [89]:
df_temp.AreaName.unique()

array(['ES CTA', 'ES BZN', 'RO CTA', 'RO CTY', 'HR CTY', 'HU CTA',
       'DE CTY', 'DE(Amprion) CTA', 'CZ CTY', 'DE(TransnetBW) CTA',
       'DE(50Hertz) CTA', 'HR BZN', 'HU BZN', 'DE-LU BZN', 'FI CTY',
       'ES CTY', 'DE(TenneT DE) CTA', 'FI CTA', 'FI BZN', 'CZ BZN',
       'RO BZN', 'HR CTA', 'HU CTY', 'CZ CTA', 'NL CTY', 'NL CTA',
       'AT CTY', 'LV CTA', 'AT CTA', 'AT BZN', 'NL BZN', 'PT CTY',
       'BE BZN', 'PT BZN', 'LU CTA', 'LV CTY', 'LV BZN', 'PL BZN',
       'LU CTY', 'EE BZN', 'LT BZN', 'PT CTA', 'EE CTA', 'EE CTY',
       'GE CTA', 'GE BZN', 'GE CTY', 'PL CTA', 'PL CTY', 'XK CTA',
       'XK BZN', 'SK CTA', 'XK CTY', 'BE CTA', 'BE CTY', 'LT CTA',
       'LT CTY', 'ME BZN', 'NIE CTA', 'IT-Calabria BZN', 'IT-South BZN',
       'SK BZN', 'SK CTY', 'ME CTA', 'ME CTY', 'UK CTY', 'GR CTA',
       'IT-North BZN', 'IT-Sardinia BZN', 'FR BZN', 'NO3 BZN', 'SI BZN',
       'DK CTY', 'GR CTY', 'GR BZN', 'IT-Sicily BZN',
       'IT-Centre-South BZN', 'IT CTY', 'BA BZN', 'BA CTY',

In [90]:
df_temp.MapCode.unique()

array(['ES', 'RO', 'HR', 'HU', 'DE', 'DE_Amprion', 'CZ', 'DE_TransnetBW',
       'DE_50HzT', 'DE_LU', 'FI', 'DE_TenneT_GER', 'NL', 'AT', 'LV', 'PT',
       'BE', 'LU', 'PL', 'EE', 'LT', 'GE', 'XK', 'SK', 'ME', 'NIE',
       'IT-Calabria', 'IT-SOUTH', 'GB', 'GR', 'IT-NORTH', 'IT-Sardinia',
       'FR', 'NO3', 'SI', 'DK', 'IT-Sicily', 'IT-CSOUTH', 'IT', 'BA',
       'IT-CNORTH', 'NO5', 'NO2', 'NO', 'DK2', 'SE3', 'SE1', 'SE', 'CH',
       'RS', 'SE2', 'NO1', 'SE4', 'NO4', 'DK1', 'BG', 'MK', 'IE',
       'IE_SEM', 'MD'], dtype=object)

In [91]:
# Ungewollte Spalten beseitigen: (Leon)

df_gen_in = df_gen_in.drop(columns =['ResolutionCode','AreaName','ProductionType','UpdateTime'])
df_gen_in.head()                    

,DateTime,MapCode,ActualGenerationOutput,ActualConsumption,technology
2418,2024-01-01,PL,0.00,NaN,Reservoir
3553,2024-01-01,RS,243.94,NaN,WindOnshore
3554,2024-01-01,RS,27.00,NaN,Other
3555,2024-01-01,RS,30.00,NaN,Biomass
9459,2024-01-01,BG,116.11,NaN,WindOnshore


In [99]:
#Aggregate technologies:
df_gen = df_gen_in.groupby(["DateTime", "MapCode", "technology"], as_index=False).sum() # Leon, durch .sum() werden Nuller bei actual consumption angeziegt, obwohl das NaN Werte sind 
df_gen.columns = ["date", "country", "tech", "output", "demand"]
df_gen["net_generation"] = df_gen['output'] - df_gen['demand']
df_gen.info(show_counts=True) # new in pandas, Leon 
df_gen.head()

,DateTime,MapCode,technology,ActualGenerationOutput,ActualConsumption
0,2024-01-01,AT,Biomass,144.00,0.0
1,2024-01-01,AT,Gas,32.80,0.0
2,2024-01-01,AT,HardCoal,0.00,0.0
3,2024-01-01,AT,Oil,0.00,0.0
4,2024-01-01,AT,Other,122.07,0.0


In [93]:
df_gen = df_gen.set_index(['date','country','tech'])
df_gen.head()

output  demand  net_generation
date       country tech                                    
2024-01-01 AT      Biomass   144.00     0.0          144.00
                   Gas        32.80     0.0           32.80
                   HardCoal    0.00     0.0            0.00
                   Oil         0.00     0.0            0.00
                   Other     122.07     0.0          122.07

In [94]:
#some values are reported quarter hourly so we have to resample to hourly values
df_gen_hourly = df_gen.groupby([pd.Grouper(level='country'),
                                pd.Grouper(level='tech'), 
                                pd.Grouper(level='date', freq='1H')]
                               ).mean()
df_gen_hourly.head()

C:\Users\UDuer\AppData\Local\Temp\ipykernel_24300\3725565403.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pd.Grouper(level='date', freq='1H')]


output  demand  net_generation
country tech    date                                               
AT      Biomass 2024-01-01 00:00:00   143.0     0.0           143.0
                2024-01-01 01:00:00   140.0     0.0           140.0
                2024-01-01 02:00:00   140.0     0.0           140.0
                2024-01-01 03:00:00   140.0     0.0           140.0
                2024-01-01 04:00:00   140.0     0.0           140.0

In [69]:
#We are primarily interested in renewable data. Let's see how complete they are
df_ = df_gen_hourly.groupby(["country", "tech"]).net_generation.count()
df_ = df_.reset_index().pivot_table(index=["country"], columns="tech", values="net_generation")
df_.loc[(slice(None)), ["Solar", "WindOnshore", "WindOffshore", "RunOfRiver","Biomass"]]

tech,Solar,WindOnshore,WindOffshore,RunOfRiver,Biomass
country,,,,,
AT,8784.0,8784.0,NaN,8784.0,8784.0
BA,7849.0,7849.0,NaN,7849.0,NaN
BE,8784.0,8784.0,8784.0,8784.0,8784.0
BG,8784.0,8784.0,NaN,8784.0,8784.0
CH,8784.0,8784.0,NaN,8784.0,NaN
CZ,8783.0,8784.0,NaN,8784.0,8784.0
DE,8784.0,8784.0,8784.0,8784.0,8784.0
DK,8783.0,8784.0,8783.0,NaN,8783.0
EE,8784.0,8783.0,NaN,8783.0,8783.0


In [70]:
#export annual and monthly data for checking of the data quality
df_gen_a = df_gen_hourly.groupby(["country", "tech"]).sum()/1000000
df_gen_a.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 293 entries, ('AT', 'Biomass') to ('XK', 'Lignite')
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   output          293 non-null    float64
 1   demand          293 non-null    float64
 2   net_generation  293 non-null    float64
dtypes: float64(3)
memory usage: 7.9+ KB


In [71]:
df_gen_m = df_gen_hourly.groupby([pd.Grouper(freq='M', level='date'), "country", "tech"]).sum()
df_gen_m.info()

C:\Users\UDuer\AppData\Local\Temp\ipykernel_24300\2149469722.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_gen_m = df_gen_hourly.groupby([pd.Grouper(freq='M', level='date'), "country", "tech"]).sum()


<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3484 entries, (Timestamp('2024-01-31 00:00:00'), 'AT', 'Biomass') to (Timestamp('2024-12-31 00:00:00'), 'XK', 'Lignite')
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   output          3484 non-null   float64
 1   demand          3484 non-null   float64
 2   net_generation  3484 non-null   float64
dtypes: float64(3)
memory usage: 92.5+ KB


In [72]:
#we also export weekly values
df_gen_w = df_gen_hourly.groupby([pd.Grouper(freq='W', level='date'), "country", "tech"]).sum()
df_gen_w.head()

output  demand  net_generation
date       country tech                                       
2024-01-07 AT      Biomass    24453.00     0.0        24453.00
                   Gas       123265.00     0.0       123265.00
                   HardCoal       0.00     0.0            0.00
                   Oil            0.00     0.0            0.00
                   Other      20507.76     0.0        20507.76

In [73]:
df_gen_hourly.to_csv(dir_out + "generation_"+year+"_hourly_entsoe.csv", index=True)
df_gen_w.to_csv(dir_out + "generation_"+year+"_weekly_entsoe.csv", index=True)
df_gen_a.to_csv(dir_out + "generation_"+year+"_annual_entsoe.csv", index=True)
df_gen_m.to_csv(dir_out + "generation_"+year+"_monthly_entsoe.csv", index=True)

OSError: Cannot save file into a non-existent directory: '..\parsed_data'

In [22]:
df_gen.reset_index().country.unique()

NameError: name 'df_gen' is not defined